In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Using a relative path from your notebook's location to the data file
file_path = "Input_Data/01_cleaned_data/uwb_dataset_02a_normalized_class_data.parquet"

# Load the parquet file into a DataFrame
df = pd.read_parquet(file_path)

# Step 2: Separate Targets (Y) from Features (X)
# We do not want to scale the 'NLOS' label or the 'RANGE' target
target_columns = ['NLOS', 'RANGE']

# Get all other columns to act as our features (FP metrics, Noise, RXPACC, and CIRs)
feature_columns = [col for col in df.columns if col not in target_columns]

X = df[feature_columns]
Y = df[target_columns]

# Step 3: Initialize the Scaler
# StandardScaler transforms data to have a mean of 0 and a standard deviation of 1.
# This is generally the best choice for PCA, SVMs, and Neural Networks.
scaler = StandardScaler()

# Step 4: Fit and Transform the features
# The scaler learns the min/max/mean of the data (fit) and applies the math (transform)
X_scaled_array = scaler.fit_transform(X)

# Step 5: Convert the scaled array back into a pandas DataFrame
# fit_transform outputs a raw numpy array, so we put the column names back on
X_scaled = pd.DataFrame(X_scaled_array, columns=feature_columns)

# Step 6: (Optional) Recombine the unscaled targets with the scaled features
# This gives you a clean, fully prepared dataset for the next steps
df_prepared = pd.concat([Y, X_scaled], axis=1)

# Display the first few rows to verify
print("Original FP_AMP1 values:", X['FP_AMP1'].head(3).values)
print("Scaled FP_AMP1 values:  ", X_scaled['FP_AMP1'].head(3).values)
print("\nPrepared Dataset Shape:", df_prepared.shape)

In [ ]:
import pandas as pd
import numpy as np

# Make a copy of the original dataframe to keep it safe
df_clean = df.copy()
print(f"Original Dataset Shape: {df_clean.shape}\n")

# ==========================================
# 1. OUTLIER REMOVAL (Using IQR Method)
# ==========================================
print("--- 1. Outlier Removal ---")
# We will check the hardware noise columns. You can add 'CIR_PWR' or 'RXPACC' to this list if needed.
columns_to_check = ['STDEV_NOISE', 'MAX_NOISE'] 

for col in columns_to_check:
    if col in df_clean.columns:
        # Calculate Q1 (25th percentile) and Q3 (75th percentile)
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        
        # Calculate Interquartile Range (IQR)
        IQR = Q3 - Q1

        # Define bounds for what is considered an "outlier" (1.5x IQR is the statistical standard)
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Count how many outliers exist before removing them
        outliers_count = len(df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)])
        print(f"Found {outliers_count} extreme outliers in '{col}'.")

        # Filter the dataframe to KEEP only rows within the normal bounds
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

print(f"Dataset Shape after Outlier Removal: {df_clean.shape}\n")


# ==========================================
# 2. DATA LEAKAGE CHECK (Using Correlation)
# ==========================================
print("--- 2. Data Leakage Check ---")
# Data leakage usually happens if a feature is basically a direct mathematical proxy for your target.
# We test this by checking the Pearson Correlation.

# To keep the calculation fast and readable, we will exclude the 1000+ raw CIR columns for this check
non_cir_cols = [c for c in df_clean.columns if not c.startswith('CIR') or c == 'CIR_PWR']

# Calculate correlation matrix for the non-CIR columns
correlation_matrix = df_clean[non_cir_cols].corr()

# Check correlations against the regression target: RANGE
if 'RANGE' in correlation_matrix.columns:
    # Get the absolute correlation values, sort them highest to lowest
    range_corr = correlation_matrix['RANGE'].abs().sort_values(ascending=False)
    
    print("Top features most correlated with the target 'RANGE':")
    # We skip the 1st one (.head(6)[1:]) because RANGE always has a 1.0 correlation with itself
    print(range_corr.head(6)[1:]) 
    
    # Check for dangerously high correlations (Greater than 95%)
    potential_leaks = range_corr[(range_corr > 0.95) & (range_corr < 1.0)]
    
    if not potential_leaks.empty:
        print("\n⚠️ WARNING: Potential Data Leakage Detected!")
        print("These features have > 0.95 correlation with RANGE. If they are distance estimates rather than raw hardware metrics, you must drop them before training:")
        print(potential_leaks)
    else:
        print("\n✅ Good news: No obvious data leakage detected (No features correlate > 0.95 with RANGE).")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

# 1. Define our columns (Assuming 'df_clean' is your dataset without outliers)
targets = ['NLOS', 'RANGE']
cir_cols = [c for c in df_clean.columns if c.startswith('CIR') and c != 'CIR_PWR']
standard_cols = [c for c in df_clean.columns if c not in targets and c not in cir_cols]

# 2. Separate Features (X) and Target (y)
X = df_clean[standard_cols + cir_cols]
y = df_clean['NLOS'] # We are classifying NLOS

# Split the data into Training and Testing sets (80% train, 20% test)
# stratify=y ensures we have a balanced number of LOS/NLOS in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ==========================================
# 3. BUILD THE PREPROCESSING PIPELINE
# ==========================================

# A. What happens to the CIR columns: Scale them, then apply PCA
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA()) # We leave n_components blank; the Grid Search will fill it in!
])

# B. What happens to standard columns (STDEV_NOISE, FP_AMP1, etc.): Only Scale them
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# C. Combine them using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

# ==========================================
# 4. BUILD THE FULL MODEL PIPELINE
# ==========================================
# This connects the preprocessor directly to our proxy classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))
])

# ==========================================
# 5. RUN THE GRID SEARCH
# ==========================================
# We tell the grid search to test PCA components from 10 to 50
param_grid = {
    'preprocessor__cir_processing__pca__n_components': [10, 15, 20, 25, 30, 35, 40, 50]
}

print("Starting Grid Search to find optimal PCA components. This may take a minute...")

# cv=5 means it will 5-fold cross-validate every single PCA number to ensure it's not a fluke
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', verbose=1)
grid_search.fit(X_train, y_train)

# ==========================================
# 6. THE RESULTS
# ==========================================
print("\n" + "="*40)
print(f"✅ BEST PCA COMPONENTS: {grid_search.best_params_['preprocessor__cir_processing__pca__n_components']}")
print(f"✅ BEST CROSS-VAL ACCURACY: {grid_search.best_score_ * 100:.2f}%")
print("="*40 + "\n")

# If you want to see how all the other numbers performed:
results = pd.DataFrame(grid_search.cv_results_)
print("Accuracy for each number of components tested:")
for index, row in results.iterrows():
    comps = row['params']['preprocessor__cir_processing__pca__n_components']
    acc = row['mean_test_score'] * 100
    print(f" - {comps:2d} components: {acc:.2f}% accuracy")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

# Import Classifiers
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# ==========================================
# 1. LOCK IN THE OPTIMAL PREPROCESSOR
# ==========================================
OPTIMAL_PCA_COMPONENTS = grid_search.best_params_['preprocessor__cir_processing__pca__n_components']

print(f"Building preprocessor with {OPTIMAL_PCA_COMPONENTS} CIR PCA components...\n")

cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# (Assumes cir_cols and standard_cols are still stored in memory from your previous code)
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])



In [ ]:
from sklearn.model_selection import GridSearchCV

# ==========================================
# 2. DEFINE MODELS AND THEIR HYPERPARAMETER GRIDS
# ==========================================
# Instead of just the model, we now define a grid of parameters to test for each one.
# Note: The keys in the param_grid must start with 'classifier__' so the pipeline knows 
# these settings belong to the model, not the PCA preprocessor.

models_and_params = {
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {
            'classifier__n_estimators': [50, 100],
            'classifier__max_depth': [10, 20, None]
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            'classifier__max_depth': [5, 10, 15, None],
            'classifier__min_samples_split': [2, 5, 10]
        }
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            'classifier__n_neighbors': [3, 5, 7, 11],
            'classifier__weights': ['uniform', 'distance']
        }
    },
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=2000, random_state=42),
        "params": {
            'classifier__C': [0.01, 0.1, 1.0, 10.0]
        }
    },
    "Naive Bayes": {
        "model": GaussianNB(),
        "params": {
            'classifier__var_smoothing': [1e-9, 1e-7, 1e-5, 1e-3]
        }
    },
    "SVM (RBF Kernel)": {
        "model": SVC(kernel='rbf', probability=True, random_state=42),
        "params": {
            'classifier__C': [0.1, 1.0, 10.0],
            'classifier__gamma': ['scale', 'auto']
        }
    }
}

# Variables to store results for graphing
train_accuracies = []
test_accuracies = []
model_names = []
results_dict = {}
best_estimators = {} # We will save the perfectly tuned models here

print("Tuning, Training, and Evaluating models. This may take a few minutes...\n")

In [ ]:
# ==========================================
# STEP 3: PREPROCESSING PIPELINE
# ==========================================
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Lock in the optimal number discovered in the previous grid search
OPTIMAL_PCA_COMPONENTS = 25

print(f"Initializing preprocessor with {OPTIMAL_PCA_COMPONENTS} PCA components...")

# Pipeline for raw CIR Waveform data
cir_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=OPTIMAL_PCA_COMPONENTS)) 
])

# Pipeline for standard hardware metrics
standard_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Combine them: This ensures the right math is applied to the right columns
preprocessor = ColumnTransformer(transformers=[
    ('cir_processing', cir_transformer, cir_cols),
    ('std_processing', standard_transformer, standard_cols)
])

print("✅ Preprocessing pipeline ready.")

In [ ]:
# ==========================================
# SETUP: IMPORTS & GLOBAL STORAGE
# ==========================================
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.pipeline import Pipeline
# Assuming your specific models (RandomForestClassifier, SVC, etc.) and 'preprocessor' are already imported/defined
import matplotlib.pyplot as plt
import pandas as pd

# Global storage for the final combined ROC curve
roc_data = {}

In [ ]:
# ==========================================
# MODEL 1: RANDOM FOREST
# ==========================================
print("Tuning and Evaluating: Random Forest...")
model_name_rf = "Random Forest"

# 1. Setup Model & Pipeline
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_params = {'classifier__n_estimators': [50, 100], 'classifier__max_depth': [10, 20, None]}
pipeline_rf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', rf_model)])

# 2. Grid Search
grid_search_rf = GridSearchCV(pipeline_rf, rf_params, cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid_search_rf.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_rf = grid_search_rf.best_estimator_
y_pred_rf = best_model_rf.predict(X_test)
y_proba_rf = best_model_rf.predict_proba(X_test)[:, 1]

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_rf, axes_rf = plt.subplots(1, 3, figsize=(18, 5))
fig_rf.suptitle(f'Comprehensive Evaluation: {model_name_rf}', fontsize=16, fontweight='bold')

# Graph 1: Tuning
cv_results_rf = pd.DataFrame(grid_search_rf.cv_results_)
param_labels_rf = [str(p).replace('classifier__', '') for p in cv_results_rf['params']]
axes_rf[0].plot(param_labels_rf, cv_results_rf['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
axes_rf[0].plot(param_labels_rf, cv_results_rf['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
axes_rf[0].set_title(f'{model_name_rf}: Tuning Results')
axes_rf[0].set_ylabel('Accuracy (%)')
axes_rf[0].tick_params(axis='x', rotation=45)
axes_rf[0].legend()
axes_rf[0].grid(True, alpha=0.3)

# Graph 2: Confusion Matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=best_model_rf.classes_)
disp_rf.plot(ax=axes_rf[1], cmap='Blues', colorbar=False)
axes_rf[1].set_title(f'{model_name_rf}: Confusion Matrix')

# Graph 3: Individual ROC Curve
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)
roc_data[model_name_rf] = {'fpr': fpr_rf, 'tpr': tpr_rf, 'auc': roc_auc_rf} # Save for combined graph

axes_rf[2].plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f'AUC = {roc_auc_rf:.3f}')
axes_rf[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes_rf[2].set_xlim([0.0, 1.0])
axes_rf[2].set_ylim([0.0, 1.05])
axes_rf[2].set_xlabel('False Positive Rate')
axes_rf[2].set_ylabel('True Positive Rate')
axes_rf[2].set_title(f'{model_name_rf}: ROC Curve')
axes_rf[2].legend(loc="lower right")

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# ==========================================
# MODEL 2: DECISION TREE
# ==========================================
print("Tuning and Evaluating: Decision Tree...")
model_name_dt = "Decision Tree"

# 1. Setup Model & Pipeline
dt_model = DecisionTreeClassifier(random_state=42)
dt_params = {'classifier__max_depth': [5, 10, 20, None]}
pipeline_dt = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', dt_model)])

# 2. Grid Search
grid_search_dt = GridSearchCV(pipeline_dt, dt_params, cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid_search_dt.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_dt = grid_search_dt.best_estimator_
y_pred_dt = best_model_dt.predict(X_test)
y_proba_dt = best_model_dt.predict_proba(X_test)[:, 1]

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_dt, axes_dt = plt.subplots(1, 3, figsize=(18, 5))
fig_dt.suptitle(f'Comprehensive Evaluation: {model_name_dt}', fontsize=16, fontweight='bold')

# Graph 1: Tuning
cv_results_dt = pd.DataFrame(grid_search_dt.cv_results_)
param_labels_dt = [str(p).replace('classifier__', '') for p in cv_results_dt['params']]
axes_dt[0].plot(param_labels_dt, cv_results_dt['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
axes_dt[0].plot(param_labels_dt, cv_results_dt['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
axes_dt[0].set_title(f'{model_name_dt}: Tuning Results')
axes_dt[0].set_ylabel('Accuracy (%)')
axes_dt[0].tick_params(axis='x', rotation=45)
axes_dt[0].legend()
axes_dt[0].grid(True, alpha=0.3)

# Graph 2: Confusion Matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)
disp_dt = ConfusionMatrixDisplay(confusion_matrix=cm_dt, display_labels=best_model_dt.classes_)
disp_dt.plot(ax=axes_dt[1], cmap='Blues', colorbar=False)
axes_dt[1].set_title(f'{model_name_dt}: Confusion Matrix')

# Graph 3: Individual ROC Curve
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_proba_dt)
roc_auc_dt = auc(fpr_dt, tpr_dt)
roc_data[model_name_dt] = {'fpr': fpr_dt, 'tpr': tpr_dt, 'auc': roc_auc_dt}

axes_dt[2].plot(fpr_dt, tpr_dt, color='darkorange', lw=2, label=f'AUC = {roc_auc_dt:.3f}')
axes_dt[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes_dt[2].set_xlim([0.0, 1.0])
axes_dt[2].set_ylim([0.0, 1.05])
axes_dt[2].set_xlabel('False Positive Rate')
axes_dt[2].set_ylabel('True Positive Rate')
axes_dt[2].set_title(f'{model_name_dt}: ROC Curve')
axes_dt[2].legend(loc="lower right")

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# ==========================================
# MODEL 3: KNN
# ==========================================
print("Tuning and Evaluating: KNN...")
model_name_knn = "KNN"

# 1. Setup Model & Pipeline
knn_model = KNeighborsClassifier()
knn_params = {'classifier__n_neighbors': [3, 5, 11, 21]}
pipeline_knn = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', knn_model)])

# 2. Grid Search
grid_search_knn = GridSearchCV(pipeline_knn, knn_params, cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid_search_knn.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_knn = grid_search_knn.best_estimator_
y_pred_knn = best_model_knn.predict(X_test)
y_proba_knn = best_model_knn.predict_proba(X_test)[:, 1]

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_knn, axes_knn = plt.subplots(1, 3, figsize=(18, 5))
fig_knn.suptitle(f'Comprehensive Evaluation: {model_name_knn}', fontsize=16, fontweight='bold')

# Graph 1: Tuning
cv_results_knn = pd.DataFrame(grid_search_knn.cv_results_)
param_labels_knn = [str(p).replace('classifier__', '') for p in cv_results_knn['params']]
axes_knn[0].plot(param_labels_knn, cv_results_knn['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
axes_knn[0].plot(param_labels_knn, cv_results_knn['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
axes_knn[0].set_title(f'{model_name_knn}: Tuning Results')
axes_knn[0].set_ylabel('Accuracy (%)')
axes_knn[0].tick_params(axis='x', rotation=45)
axes_knn[0].legend()
axes_knn[0].grid(True, alpha=0.3)

# Graph 2: Confusion Matrix
cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=best_model_knn.classes_)
disp_knn.plot(ax=axes_knn[1], cmap='Blues', colorbar=False)
axes_knn[1].set_title(f'{model_name_knn}: Confusion Matrix')

# Graph 3: Individual ROC Curve
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_proba_knn)
roc_auc_knn = auc(fpr_knn, tpr_knn)
roc_data[model_name_knn] = {'fpr': fpr_knn, 'tpr': tpr_knn, 'auc': roc_auc_knn}

axes_knn[2].plot(fpr_knn, tpr_knn, color='darkorange', lw=2, label=f'AUC = {roc_auc_knn:.3f}')
axes_knn[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes_knn[2].set_xlim([0.0, 1.0])
axes_knn[2].set_ylim([0.0, 1.05])
axes_knn[2].set_xlabel('False Positive Rate')
axes_knn[2].set_ylabel('True Positive Rate')
axes_knn[2].set_title(f'{model_name_knn}: ROC Curve')
axes_knn[2].legend(loc="lower right")

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# ==========================================
# MODEL 4: LOGISTIC REGRESSION
# ==========================================
print("Tuning and Evaluating: Logistic Regression...")
model_name_lr = "Logistic Regression"

# 1. Setup Model & Pipeline
lr_model = LogisticRegression(max_iter=2000, random_state=42)
lr_params = {'classifier__C': [0.1, 1.0, 10.0]}
pipeline_lr = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', lr_model)])

# 2. Grid Search
grid_search_lr = GridSearchCV(pipeline_lr, lr_params, cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid_search_lr.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_lr = grid_search_lr.best_estimator_
y_pred_lr = best_model_lr.predict(X_test)
y_proba_lr = best_model_lr.predict_proba(X_test)[:, 1]

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_lr, axes_lr = plt.subplots(1, 3, figsize=(18, 5))
fig_lr.suptitle(f'Comprehensive Evaluation: {model_name_lr}', fontsize=16, fontweight='bold')

# Graph 1: Tuning
cv_results_lr = pd.DataFrame(grid_search_lr.cv_results_)
param_labels_lr = [str(p).replace('classifier__', '') for p in cv_results_lr['params']]
axes_lr[0].plot(param_labels_lr, cv_results_lr['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
axes_lr[0].plot(param_labels_lr, cv_results_lr['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
axes_lr[0].set_title(f'{model_name_lr}: Tuning Results')
axes_lr[0].set_ylabel('Accuracy (%)')
axes_lr[0].tick_params(axis='x', rotation=45)
axes_lr[0].legend()
axes_lr[0].grid(True, alpha=0.3)

# Graph 2: Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=best_model_lr.classes_)
disp_lr.plot(ax=axes_lr[1], cmap='Blues', colorbar=False)
axes_lr[1].set_title(f'{model_name_lr}: Confusion Matrix')

# Graph 3: Individual ROC Curve
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
roc_auc_lr = auc(fpr_lr, tpr_lr)
roc_data[model_name_lr] = {'fpr': fpr_lr, 'tpr': tpr_lr, 'auc': roc_auc_lr}

axes_lr[2].plot(fpr_lr, tpr_lr, color='darkorange', lw=2, label=f'AUC = {roc_auc_lr:.3f}')
axes_lr[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes_lr[2].set_xlim([0.0, 1.0])
axes_lr[2].set_ylim([0.0, 1.05])
axes_lr[2].set_xlabel('False Positive Rate')
axes_lr[2].set_ylabel('True Positive Rate')
axes_lr[2].set_title(f'{model_name_lr}: ROC Curve')
axes_lr[2].legend(loc="lower right")

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# ==========================================
# MODEL 6: SVM (RBF)
# ==========================================
print("Tuning and Evaluating: SVM (RBF)...")
model_name_svm = "SVM (RBF)"

# 1. Setup Model & Pipeline
svm_model = SVC(kernel='rbf', probability=True, random_state=42)
svm_params = {'classifier__C': [0.1, 1.0, 10.0]}
pipeline_svm = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', svm_model)])

# 2. Grid Search
grid_search_svm = GridSearchCV(pipeline_svm, svm_params, cv=3, scoring='accuracy', n_jobs=-1, return_train_score=True)
grid_search_svm.fit(X_train, y_train)

# 3. Predict using Best Model
best_model_svm = grid_search_svm.best_estimator_
y_pred_svm = best_model_svm.predict(X_test)
y_proba_svm = best_model_svm.predict_proba(X_test)[:, 1]

# ==========================================
# GRAPHING: 1x3 Dashboard
# ==========================================
fig_svm, axes_svm = plt.subplots(1, 3, figsize=(18, 5))
fig_svm.suptitle(f'Comprehensive Evaluation: {model_name_svm}', fontsize=16, fontweight='bold')

# Graph 1: Tuning
cv_results_svm = pd.DataFrame(grid_search_svm.cv_results_)
param_labels_svm = [str(p).replace('classifier__', '') for p in cv_results_svm['params']]
axes_svm[0].plot(param_labels_svm, cv_results_svm['mean_train_score'] * 100, label='Train Acc', marker='s', ls='--', color='skyblue')
axes_svm[0].plot(param_labels_svm, cv_results_svm['mean_test_score'] * 100, label='Test Acc (CV)', marker='o', ls='-', color='salmon')
axes_svm[0].set_title(f'{model_name_svm}: Tuning Results')
axes_svm[0].set_ylabel('Accuracy (%)')
axes_svm[0].tick_params(axis='x', rotation=45)
axes_svm[0].legend()
axes_svm[0].grid(True, alpha=0.3)

# Graph 2: Confusion Matrix
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(confusion_matrix=cm_svm, display_labels=best_model_svm.classes_)
disp_svm.plot(ax=axes_svm[1], cmap='Blues', colorbar=False)
axes_svm[1].set_title(f'{model_name_svm}: Confusion Matrix')

# Graph 3: Individual ROC Curve
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_proba_svm)
roc_auc_svm = auc(fpr_svm, tpr_svm)
roc_data[model_name_svm] = {'fpr': fpr_svm, 'tpr': tpr_svm, 'auc': roc_auc_svm}

axes_svm[2].plot(fpr_svm, tpr_svm, color='darkorange', lw=2, label=f'AUC = {roc_auc_svm:.3f}')
axes_svm[2].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes_svm[2].set_xlim([0.0, 1.0])
axes_svm[2].set_ylim([0.0, 1.05])
axes_svm[2].set_xlabel('False Positive Rate')
axes_svm[2].set_ylabel('True Positive Rate')
axes_svm[2].set_title(f'{model_name_svm}: ROC Curve')
axes_svm[2].legend(loc="lower right")

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.show()

In [ ]:
# ==========================================
# STEP 5: FINAL COMPARISON SUMMARY (FIXED)
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# 1. Dynamically rebuild the lists based on the independent cells you ran
final_model_names = []
final_train_accuracies = []
final_test_accuracies = []

# Map of the specific variable names we used in the previous cells
compiled_models = [
    ('model_name_rf', 'best_model_rf', 'y_pred_rf'),
    ('model_name_dt', 'best_model_dt', 'y_pred_dt'),
    ('model_name_knn', 'best_model_knn', 'y_pred_knn'),
    ('model_name_lr', 'best_model_lr', 'y_pred_lr'),
    ('model_name_nb', 'best_model_nb', 'y_pred_nb'),
    ('model_name_svm', 'best_model_svm', 'y_pred_svm')
]

# Safely check notebook memory to see which models actually finished running
for name_var, model_var, pred_var in compiled_models:
    if name_var in globals() and model_var in globals() and pred_var in globals():
        
        # Pull the variables from Jupyter's memory
        m_name = globals()[name_var]
        m_model = globals()[model_var]
        m_pred = globals()[pred_var]
        
        # Calculate accuracies and append to our lists
        final_model_names.append(m_name)
        final_train_accuracies.append(accuracy_score(y_train, m_model.predict(X_train)) * 100)
        final_test_accuracies.append(accuracy_score(y_test, m_pred) * 100)

# 2. Check if we found data, then Plot
if len(final_model_names) == 0:
    print("Error: No models found. Please run at least one of the model cells above.")
else:
    print(f"Generating chart for: {', '.join(final_model_names)}...\n")
    
    x = np.arange(len(final_model_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(13, 6))
    rects1 = ax.bar(x - width/2, final_train_accuracies, width, label='Best Train Accuracy', color='skyblue', edgecolor='black')
    rects2 = ax.bar(x + width/2, final_test_accuracies, width, label='Best Test Accuracy', color='salmon', edgecolor='black')

    ax.set_ylabel('Accuracy (%)', fontweight='bold')
    ax.set_title('Final Comparison: Best Tuned Models (Train vs. Test)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(final_model_names, rotation=45, ha='right')
    ax.set_ylim(0, 115)
    ax.legend(loc='lower right')
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Labeling bars
    for rect in rects1 + rects2:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%', xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()